# NumPy NN - NumPy Advanced

> **MLCourse · Data Science Foundations · 01_numpy**

Level two of the NumPy track: the selection and reshuffling tools professionals use daily - fancy
indexing, sorting/ranking, joins and splits - plus linear algebra, the modern random Generator,
a deep proof-driven look at views vs copies, and the vectorization mindset that makes code fast.

## What you'll learn

- Fancy (integer-array) indexing in 1-D and 2-D, including duplicate-index gotchas
- The boolean toolkit deep cut: `np.where`, `np.clip`, `count_nonzero`, `nonzero`, `any`/`all`
- Sorting: `np.sort` vs `.sort()`, ranking with `argsort`, top-k, and O(n) `np.partition`
- `np.unique` with counts/first-occurrence indices; set operations (`intersect1d`, `isin`, ...)
- Joining (`concatenate`, `vstack`, `hstack`, `column_stack`, `dstack`) and splitting arrays
- `tile` vs `repeat` - pattern duplication done right
- Universal functions: vectorized custom formulas (compound interest, distance matrices)
- Linear algebra essentials: `@`, `trace`, `det`, `inv`, `solve`, `norm`, `eigvalsh`
- Modern random numbers with `np.random.default_rng(42)` (and why legacy calls are old style)
- Views vs copies PROVEN with memory-sharing experiments
- Performance engineering: benchmarks, why vectorization wins, avoiding array-growth antipatterns
- Saving/loading: `.npy`, `.npz`, and text round-trips

## 0. Setup

Everything here needs only NumPy plus a few standard-library helpers: `os`/`tempfile` for the
file-IO section at the end, and `time` for honest wall-clock benchmarks.

> 💡 **Pro tip:** this notebook assumes you finished `01_numpy_foundations`. If any term there
felt shaky (views, broadcasting rules, axis semantics), revisit it first - we build directly on it.

In [1]:
import tempfile                  # cross-platform scratch directory for our saved files
import os                        # path joining + cleanup afterwards
import time                      # high-resolution benchmarking timer

import numpy as np               # still just one real dependency

print("NumPy version:", np.__version__)

NumPy version: 2.4.6


## 1. Fancy indexing: selecting with integer arrays

Instead of a single index or slice, you can pass an ARRAY (or list) of indices. NumPy picks those
positions in the order given, repeats allowed, negatives allowed - and always returns a COPY
(contrast with slices-as-views from the foundations notebook).

In [2]:
a = np.array([10, 20, 30, 40, 50])        # the data

print("a[[3, 0]]      ->", a[[3, 0]])     # arbitrary order
print("a[[0, 0, 3]]   ->", a[[0, 0, 3]])  # duplicates are fine - each is an independent pick
print("a[[-1, 1]]     ->", a[[-1, 1]])   # negative indices count from the end

# Fancy indexing is also assignment target:
b = a.copy()
b[[0, 4]] = -1                            # scatter-write into several slots at once
print("after b[[0,4]]=-1 ->", b)

a[[3, 0]]      -> [40 10]
a[[0, 0, 3]]   -> [10 10 40]
a[[-1, 1]]     -> [50 20]
after b[[0,4]]=-1 -> [-1 20 30 40 -1]


> ⚠️ **Common pitfall:** accumulate-with-duplicates does NOT do what it looks like. `counts[idx]
+= 1` reads the ORIGINAL values once per slot group, so repeated indices only increment ONCE.
The unbuffered fix is `np.add.at`.

In [3]:
counts = np.zeros(5, dtype=int)
counts[[0, 0, 1]] += 1                    # looks like index 0 gets +2... it doesn't!
print("naive   += :", counts)             # [1 1 0 0 0] <- lost an increment

np.add.at(counts, [0, 0, 1], 1)           # np.add.at is UNBUFFERED: every hit lands
print("np.add.at  :", counts)             # [2 1 0 0 0]

naive   += : [1 1 0 0 0]
np.add.at  : [3 2 0 0 0]


In [4]:
# 2-D fancy indexing has TWO modes:
G = np.arange(16).reshape(4, 4)           # a 4x4 grid to raid
print("G:\n", G)

# Mode A: select whole ROWS in a chosen order:
print("rows [3,1,2]:\n", G[[3, 1, 2]])

# Mode B: pair up row+col indices ELEMENTWISE -> individual cells:
cells = G[[0, 1, 2], [1, 2, 3]]           # picks (0,1), (1,2), (2,3)
print("paired cells:", cells)

# Mixing a slice with fancy indexing selects columns across all rows:
cols = G[:, [2, 0]]                       # every row, but columns swapped
print("all rows, cols [2,0]:\n", cols)

G:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]
rows [3,1,2]:
 [[12 13 14 15]
 [ 4  5  6  7]
 [ 8  9 10 11]]
paired cells: [ 1  6 11]
all rows, cols [2,0]:
 [[ 2  0]
 [ 6  4]
 [10  8]
 [14 12]]


## 2. The boolean toolkit, deep edition

Beyond plain filtering, four functions turn masks into answers: `where` (branch or locate),
`clip` (clamp), `count_nonzero` (count), `nonzero` (coordinates of True), plus `any`/`all`
(existential/universal checks).

In [5]:
vals = np.array([88, 92, 79, 45, 100, 67, 73, 58, 95, 81])    # exam scores

# --- np.where with THREE arguments = vectorized if/else ---------------------------
pass_fail = np.where(vals >= 60, "pass", "fail")              # elementwise branch
print("labels      :", pass_fail)

# --- np.where with ONE argument = where are the Trues? ----------------------------
print("failing idx :", np.where(vals < 60)[0])                # -> array([3]) ... coordinates
print("nonzero twin:", np.nonzero(vals < 60)[0])                 # identical result

# --- np.clip clamps values into [lo, hi] -------------------------------------------
noisy = np.array([-12., 5., 48., 130., 77.])
clamped = np.clip(noisy, 0, 100)                              # outliers pulled to the edges
print("clipped     :", clamped)

# --- counting Trues two ways --------------------------------------------------------
mask_even = vals % 2 == 0
print("count_nonzero:", np.count_nonzero(mask_even), "| mask.sum():", mask_even.sum())

# --- any / all collapse a mask to one boolean ---------------------------------------
print("any >= 90?  :", bool((vals >= 90).any()))                   # does ANY element qualify?
print("all >= 45?  :",(vals >= 45).all())                          # do ALL elements qualify?

labels      : ['pass' 'pass' 'pass' 'fail' 'pass' 'pass' 'pass' 'fail' 'pass' 'pass']
failing idx : [3 7]
nonzero twin: [3 7]
clipped     : [  0.   5.  48. 100.  77.]
count_nonzero: 4 | mask.sum(): 4
any >= 90?  : True
all >= 45?  : True


> 💡 **Pro tip:** `(vals >= 90).sum()` counts matches, `.argmax()` on a BOOLEAN mask returns the
FIRST True position (or 0 when none match - check `.any()` first!). These two one-liners cover
most quick data questions without sorting anything.

> ⚠️ **Common pitfall:** `np.where(cond)` returns a TUPLE of index arrays (one per dimension).
For 1-D use `np.where(cond)[0]`; forgetting `[0]` is a classic source of shape bugs downstream.

## 3. Sorting and ranking

Three distinct jobs, three distinct tools:

| job                                  | tool                     |
|--------------------------------------|--------------------------|
| get sorted VALUES                    | `np.sort(x)` / `x.sort()`|
| get the ORDER that would sort        | `np.argsort(x)`          |
| get top/bottom-k FAST (unordered)    | `np.partition(x, k)`     |

In [6]:
raw = np.array([23, 7, 13, 42, 4, 19])

sorted_copy = np.sort(raw)            # FUNCTION form: returns a new sorted array
print("np.sort ->", sorted_copy, "| original untouched:", raw)

inplace = raw.copy()
inplace.sort()                        # METHOD form: sorts IN PLACE, returns None!
print(".sort() ->", inplace)

# argsort gives PERMUTATION indices: order[i] tells which original element lands at position i
names = np.array(["ada", "bob", "cy", "dee", "eli", "fay"])
order = np.argsort(raw)               # ascending by score
print("argsort      :", order)
print("sorted names :", names[order])       # reordering via fancy indexing!

descending = np.argsort(-raw)         # negate values to reverse (works for floats too)
print("desc order   :", descending)

np.sort -> [ 4  7 13 19 23 42] | original untouched: [23  7 13 42  4 19]
.sort() -> [ 4  7 13 19 23 42]
argsort      : [4 1 2 5 0 3]
sorted names : ['eli' 'bob' 'cy' 'fay' 'ada' 'dee']
desc order   : [3 0 5 2 1 4]


### The top-k trick

`argsort[-k:]` grabs the k LARGEST positions - reversed for highest-first presentation. This is
the single most reused ranking idiom in data science ("top-5 products", "10 worst residuals").

In [7]:
k = 3
top_idx = np.argsort(raw)[-k:][::-1]          # last k of ascending order, then flip
print("top", k, "values:", raw[top_idx], "at indices", top_idx)

# np.partition: O(n) - puts the k-th smallest in its final slot, smaller left, bigger right,
# WITHOUT fully sorting either side. Perfect when you need "top-k" but not their internal order.
part = np.partition(raw, len(raw) - k)        # everything >= 3rd largest pushed to the right
print("partitioned   :", part)
print("top-k via part:", part[-k:])          # same SET as above, unordered

top 3 values: [42 23 19] at indices [3 0 5]
partitioned   : [ 4  7 13 19 23 42]
top-k via part: [19 23 42]


> ⚠️ **Common pitfall:** `arr.sort()` returns `None` (like Python's `list.sort`). Writing
`x = arr.sort()` silently destroys your data reference. Use `x = np.sort(arr)` for a copy.

> 💡 **Pro tip:** for ties you care about (e.g., stable leaderboard), pass
`kind="stable"` to sort/argsort so equal keys keep their original relative order.

## 4. Unique values & set operations

`np.unique` is "distinct values" plus optional bookkeeping; the `*1d` set functions compare
populations between arrays. Together they answer frequency-table and membership questions fast.

In [8]:
draws = np.array([3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5])       # some categorical-ish draws

uniq_vals, counts = np.unique(draws, return_counts=True)  # value histogram
print("unique:", uniq_vals)
print("counts:", counts)
print("most common:", uniq_vals[np.argmax(counts)])       # winner-take-all lookup

uniq_vals, first_seen = np.unique(draws, return_index=True)
print("first occurrence idx:", first_seen)                # where each value FIRST appeared

# --- set operations -----------------------------------------------------------------
a_set = np.array([1, 2, 3, 4])
b_set = np.array([3, 4, 5, 6])

print("intersect1d :", np.intersect1d(a_set, b_set))       # in BOTH
print("union1d      :", np.union1d(a_set, b_set))          # in EITHER
print("setdiff1d    :", np.setdiff1d(a_set, b_set))       # in a_set but NOT b_set
print("isin         :", np.isin(a_set, [2, 4, 6]))        # elementwise membership mask

unique: [1 2 3 4 5 6 9]
counts: [2 1 2 1 3 1 1]
most common: 5
first occurrence idx: [1 6 0 2 4 7 5]
intersect1d : [3 4]
union1d      : [1 2 3 4 5 6]
setdiff1d    : [1 2]
isin         : [False  True False  True]


## 5. Joining arrays

One general function plus ergonomic wrappers. The subtlety worth memorizing: for 1-D vectors,
`hstack` concatenates END-TO-END while `column_stack` builds COLUMNS - different results!

In [9]:
m1 = np.arange(6).reshape(2, 3)           # (2, 3)
m2 = (np.arange(6) * 10).reshape(2, 3)    # (2, 3)

print("concat axis=0 (more rows):\n", np.concatenate([m1, m2], axis=0))   # (4, 3)
print("concat axis=1 (more cols):\n", np.concatenate([m1, m2], axis=1))   # (2, 6)

print("vstack == concat axis=0:", np.array_equal(np.vstack([m1, m2]),
                                                 np.concatenate([m1, m2], axis=0)))
print("hstack == concat axis=1:", np.array_equal(np.hstack([m1, m2]),
                                                 np.concatenate([m1, m2], axis=1)))
# (np.row_stack is simply a legacy alias of vstack - same function, older name.)

x_vec = np.array([1, 2, 3])
y_vec = np.array([40, 50, 60])
print("hstack 1-D     :", np.hstack([x_vec, y_vec]))       # [ 1  2  3 40 50 60] flat!
print("column_stack   :\n", np.column_stack([x_vec, y_vec]))  # (3, 2): vectors become COLUMNS

deep = np.dstack([x_vec, y_vec])                            # stack along a NEW 3rd axis
print("dstack shape   :", deep.shape, "->", deep.ravel())  # interleaved pairs

concat axis=0 (more rows):
 [[ 0  1  2]
 [ 3  4  5]
 [ 0 10 20]
 [30 40 50]]
concat axis=1 (more cols):
 [[ 0  1  2  0 10 20]
 [ 3  4  5 30 40 50]]
vstack == concat axis=0: True
hstack == concat axis=1: True
hstack 1-D     : [ 1  2  3 40 50 60]
column_stack   :
 [[ 1 40]
 [ 2 50]
 [ 3 60]]
dstack shape   : (1, 3, 2) -> [ 1 40  2 50  3 60]


> ⚠️ **Common pitfall:** building a design matrix with `hstack([x1, x2])` on 1-D features yields
> one long row vector, not an (n_samples, n_features) table. Use `column_stack` (or `c_`) there.

> 💡 **Pro tip:** joining many pieces repeatedly in a loop copies every time (see performance
section). Collect parts in a Python list and `np.concatenate(parts)` ONCE at the end.

## 6. Splitting arrays

The inverses of stacking: carve one array into equal chunks (`vsplit`/`hsplit`) or let
`array_split` handle sizes that don't divide evenly.

In [10]:
Z = np.arange(16).reshape(4, 4)               # nice even grid
top, bottom = np.vsplit(Z, 2)                 # split ROWS into 2 blocks of 2
left, right = np.hsplit(Z, 2)                 # split COLUMNS into 2 blocks of 2
print("vsplit top:\n", top)
print("hsplit right:\n", right)

uneven = np.arange(10)
pieces = np.array_split(uneven, 3)            # 4 + 3 + 3 instead of raising an error
print("array_split sizes:", [p.size for p in pieces])

try:
    np.vsplit(np.arange(18).reshape(3, 6), 2) # 3 rows cannot split evenly into 2
except ValueError as err:
    print("uneven vsplit -> ValueError:", err)

vsplit top:
 [[0 1 2 3]
 [4 5 6 7]]
hsplit right:
 [[ 2  3]
 [ 6  7]
 [10 11]
 [14 15]]
array_split sizes: [4, 3, 3]
uneven vsplit -> ValueError: array split does not result in an equal division


## 7. Tiling vs repeating

Two duplication verbs that beginners constantly swap:

- `np.tile(A, reps)` - repeat the WHOLE array as a pattern (like copy-paste).
- `np.repeat(A, n)` - repeat EACH ELEMENT in place (element-level echo).

In [11]:
pattern = np.array([1, 2, 3])

print("tile   x3:", np.tile(pattern, 3))          # [1 2 3 1 2 3 1 2 3]
print("repeat x3:", np.repeat(pattern, 3))        # [1 1 1 2 2 2 3 3 3]

block = np.tile(pattern, (2, 2))                  # 2x2 mosaic of the whole pattern
print("tile (2,2):\n", block)

grid2d = np.array([[10, 20], [30, 40]])
print("repeat rows (axis=0):\n", np.repeat(grid2d, 2, axis=0))   # each ROW doubled

tile   x3: [1 2 3 1 2 3 1 2 3]
repeat x3: [1 1 1 2 2 2 3 3 3]
tile (2,2):
 [[1 2 3 1 2 3]
 [1 2 3 1 2 3]]
repeat rows (axis=0):
 [[10 20]
 [10 20]
 [30 40]
 [30 40]]


## 8. Universal functions (ufuncs) & vectorized formulas

A **ufunc** is a compiled, elementwise function (`np.sin`, `np.exp`, `np.sqrt`, `+`, `**`, ...).
Because they broadcast, ANY formula built from ufuncs becomes a loop-free, C-speed expression.
Strategy: write the scalar math, then apply it to whole arrays unchanged.

In [12]:
# --- Worked example 1: compound interest across a whole rate curve at once ----------
principal = 10_000.0                                   # starting capital P
rates = np.linspace(0.01, 0.10, 10)                    # annual rates r: 1% .. 10%
years, compounds_per_year = 10, 4                      # t and n

future_value = principal * (1 + rates / compounds_per_year) ** (compounds_per_year * years)
for r, fv in zip(rates, future_value):
    print(f"r={r:.2f} -> A={fv:,.0f}")

r=0.01 -> A=11,050
r=0.02 -> A=12,208
r=0.03 -> A=13,483
r=0.04 -> A=14,889
r=0.05 -> A=16,436
r=0.06 -> A=18,140
r=0.07 -> A=20,016
r=0.08 -> A=22,080
r=0.09 -> A=24,352
r=0.10 -> A=26,851


> 💡 **Pro tip:** notice the formula is IDENTICAL to the textbook scalar version - no loops, no
helper functions, no special cases. That's the vectorization mindset: keep the math, swap the
containers.

In [13]:
# --- Worked example 2: pairwise distance matrix via broadcasting --------------------
# Goal: D[i, j] = Euclidean distance between point i and point j, for ALL pairs at once.
points = np.array([[0., 0.],
                   [3., 4.],
                   [6., 8.]])                          # three 2-D points

delta = points[:, None, :] - points[None, :, :]        # (N,1,D) - (1,N,D) -> (N,N,D) differences
print("delta shape:", delta.shape)                     # every pairwise component difference

D = np.sqrt((delta ** 2).sum(axis=-1))                 # Pythagoras along the LAST axis
print(D)
print("symmetric?", np.allclose(D, D.T), "| zero diagonal?", np.allclose(np.diag(D), 0))

# Same trick in 1-D with absolute differences - remember this |xi - xj| pattern:
xs = np.array([0., 2., 5., 9.])
abs_diffs = np.abs(xs[:, None] - xs[None, :])          # (4,1)-(1,4) -> (4,4) |xi-xj| grid
print(abs_diffs)

delta shape: (3, 3, 2)
[[ 0.  5. 10.]
 [ 5.  0.  5.]
 [10.  5.  0.]]
symmetric? True | zero diagonal? True
[[0. 2. 5. 9.]
 [2. 0. 3. 7.]
 [5. 3. 0. 4.]
 [9. 7. 4. 0.]]


### np.vectorize: convenient, NOT fast

`np.vectorize` wraps a scalar-only Python function so it accepts arrays. Under the hood it still
runs a Python loop per element - pure ergonomics, zero speed-up. Reach for it when the logic is
genuinely non-mathematical (string formatting, bespoke branching); reach for REAL ufuncs when it
is arithmetic.

In [14]:
def letter_grade(score):
    """Scalar-only branching that no arithmetic ufunc can express."""
    if score >= 80:
        return "A"
    elif score >= 65:
        return "B"
    return "C"

scores = np.array([91, 72, 58, 84, 65])
grades = np.vectorize(letter_grade)(scores)            # convenience wrapper -> array output
print(grades, grades.dtype)

['A' 'B' 'C' 'A' 'B'] <U1


## 9. Linear algebra essentials

Matrix multiplication uses the `@` operator (PEP 465). Everything else lives in `np.linalg`.
Golden rule for systems: prefer `solve(A, b)` over `inv(A) @ b` - faster AND numerically safer.

In [15]:
A = np.array([[4., 2.],
              [1., 3.]])
B = np.array([[1., 0.],
              [2., 5.]])

print("A @ B (matrix product):\n", A @ B)              # row-by-column contraction
print("same as np.dot:", np.allclose(A @ B, np.dot(A, B)))
print("A * B is DIFFERENT (elementwise):\n", A * B)    # the eternal mix-up

I3 = np.eye(3)                                         # identity matrix (multiplicative neutral)
print("trace(A)   :", np.trace(A))                      # sum of diagonal = 7
print("det(A)     :", round(np.linalg.det(A), 6))       # determinant = 10
print("inv(A):\n", np.linalg.inv(A))                    # inverse exists since det != 0
print("A @ inv(A) ~ I:\n", np.round(A @ np.linalg.inv(A), 6))

A @ B (matrix product):
 [[ 8. 10.]
 [ 7. 15.]]
same as np.dot: True
A * B is DIFFERENT (elementwise):
 [[ 4.  0.]
 [ 2. 15.]]
trace(A)   : 7.0
det(A)     : 10.0
inv(A):
 [[ 0.3 -0.2]
 [-0.1  0.4]]
A @ inv(A) ~ I:
 [[ 1.  0.]
 [-0.  1.]]


### Worked example: solving a linear system Ax = b

Solve exactly this system (two equations, two unknowns):

```text
2x +  y =  5
 x + 3y = 10
```

In [16]:
A_sys = np.array([[2., 1.],          # coefficient matrix: one ROW per equation
                  [1., 3.]])
b_sys = np.array([5., 10.])          # right-hand side vector

solution = np.linalg.solve(A_sys, b_sys)   # factorizes once, solves directly (LU under the hood)
print("solution x =", solution)            # expect (1, 3)

residual = A_sys @ solution - b_sys        # verify by substituting back
print("residual  =", residual, "-> correct?", np.allclose(residual, 0))
print("inv route agrees:", np.allclose(np.linalg.inv(A_sys) @ b_sys, solution))

solution x = [1. 3.]
residual  = [0. 0.] -> correct? True
inv route agrees: True


> ⚠️ **Common pitfall:** `inv()` looks innocent but squares the condition number and wastes time
computing entries you never use. In production code, `np.linalg.solve` (single RHS) or
`np.linalg.lstsq` (overdetermined systems) should appear orders of magnitude more often than
`inv`.

In [17]:
v = np.array([3., 4.])                                # classic 3-4-5 triangle as a vector
print("|v| =", np.linalg.norm(v))                     # Euclidean length = 5

S = np.array([[2., 1.],
              [1., 3.]])                              # SYMMETRIC matrix
eigenvalues = np.linalg.eigvalsh(S)                   # eigvalsh: symmetric/hermitian fast path
print("eigenvalues:", eigenvalues)                    # real, returned ascending
print("trace == sum of eigs?", np.isclose(S.trace(), eigenvalues.sum()))

|v| = 5.0
eigenvalues: [1.38196601 3.61803399]
trace == sum of eigs? True


## 10. Random numbers, done right: `default_rng`

Since NumPy 1.17 the recommended API is a **Generator** object created with
`np.random.default_rng(seed)`. Benefits: reproducibility via explicit seeds, better algorithms
(PCG64), no hidden global state, and safe parallel use. All methods below hang off your `rng`.

In [18]:
rng = np.random.default_rng(42)                       # ONE generator, seeded once, reused

dice = rng.integers(1, 7, size=(3, 4))                # uniform ints: low INCLUSIVE..high EXCLUSIVE
print("dice rolls:\n", dice)

heights = rng.normal(loc=170, scale=10, size=5)       # Gaussian draws (mean, std)
print("heights  :", heights.round(1))

uniforms = rng.uniform(0, 1, size=3)                  # continuous U[0,1)
print("uniforms :", uniforms.round(3))

lottery = rng.choice(np.arange(1, 50), size=6, replace=False)   # sampling WITHOUT replacement
print("lottery  :", np.sort(lottery))

weighted = rng.choice(["red", "green"], size=8, p=[0.75, 0.25]) # weighted categories
print("weighted :", weighted)

# --- shuffle vs permutation ----------------------------------------------------------
deck = np.arange(1, 53)                               # a fresh deck
permuted_copy = rng.permutation(deck)                 # returns a NEW shuffled array
before = deck[:3].copy()
rng.shuffle(deck)                                     # shuffles IN PLACE, returns None!
print("permutation kept original intact:", np.array_equal(deck[:3], before))
print("shuffled in place:", not np.array_equal(deck, np.arange(1, 53)))

# --- the LEGACY global-style API (still everywhere in old tutorials) ------------------
np.random.seed(42)                                    # seeds hidden global state - old style
legacy_uniforms = np.random.rand(3)                   # np.random.* namespace = legacy
print("legacy draw:", legacy_uniforms)

dice rolls:


 [[1 5 4 3]
 [3 6 1 5]
 [2 1 4 6]]
heights  : [171.3 166.8 169.8 161.5 178.8]
uniforms : [0.927 0.644 0.823]
lottery  : [ 5 11 20 21 24 28]
weighted : ['green' 'red' 'green' 'green' 'green' 'red' 'red' 'red']
permutation kept original intact: False
shuffled in place: True
legacy draw: [0.37454012 0.95071431 0.73199394]


> ⚠️ **Common pitfall:** creating a NEW generator inside a loop -
`np.random.default_rng()` with no seed per iteration - gives unreproducible, possibly correlated
streams. Instantiate ONCE (seeded), pass the `rng` around like any other parameter.

> 💡 **Pro tip:** `Generator.integers` excludes the high value (perfect for dice: 1..7), while
`Generator.choice` includes endpoints of the provided pool. Reading the docstring beats guessing.

## 11. Views vs copies - the deep dive

Foundations showed that slices are views. Now let's PROVE the full rule set experimentally,
because silent aliasing causes the nastiest data-science bugs:

| operation                 | returns |
|---------------------------|-------------------|
| basic slice `a[:]`, `a[1:]`, `a[::2]` | VIEW  |
| `reshape`, `ravel` (when possible)    | VIEW  |
| `.T` transpose                        | VIEW  |
| fancy indexing `a[[1, 3]]`            | COPY  |
| boolean masking `a[a > 2]`            | COPY  |
| `.copy()`, `flatten()`, `astype(...)` | COPY  |

In [19]:
a = np.arange(6)                          # source array: [0 1 2 3 4 5]

# --- Experiment 1: a[:] is a view - writes leak through ------------------------------
b = a[:]
print("shares memory:", np.shares_memory(a, b))       # True - same buffer, two headers
b[0] = 999                                            # mutate through the 'copy'...
print("a changed too!:", a)                           # ...original shows 999

# --- Experiment 2: .copy() severs the link --------------------------------------------
c = a.copy()
c[:] = -7                                             # blast the independent buffer
print("a survived .copy():", a)

# --- Experiment 3: fancy indexing COPIES - mutations stay local ----------------------
f = a[[1, 2]]
f[0] = -100                                           # f is its own buffer...
print("a unaffected by fancy edit:", a)               # ...original untouched (vs Experiment 1!)

# --- Experiment 4: reshape usually views ---------------------------------------------
R = a.reshape(2, 3)
print("reshape shares memory:", np.shares_memory(a, R))
R[0, 0] += 100                                        # write through the reshaped view
print("a sees it:", a[0])                             # original updated again

shares memory: True
a changed too!: [999   1   2   3   4   5]
a survived .copy(): [999   1   2   3   4   5]
a unaffected by fancy edit: [999   1   2   3   4   5]
reshape shares memory: True
a sees it: 1099


> ⚠️ **Common pitfall:** chained slicing keeps you INSIDE view territory - `df.values[:, 0][0] = 1`
style double-indexing can silently edit shared buffers. When in doubt, ask NumPy directly:
`np.shares_memory(x, y)` (or check `x.base` - views point at their parent, copies have none).

> 💡 **Pro tip:** views are a FEATURE: filtering huge datasets into overlapping windows costs
nothing until you write. Adopt the habit "slice freely; `.copy()` deliberately."

## 12. Performance: thinking in whole arrays

Vectorization is not cosmetic - it moves the loop from interpreted Python into optimized C
(often SIMD-vectorized, processing multiple lanes per CPU instruction). Measure, don't guess:

In [20]:
def python_sum(values):
    """Sum with an explicit interpreted loop - what NumPy replaces."""
    total = 0.0
    for v in values:                  # one bytecode dispatch PER element
        total += v
    return total


def best_of(fn, arg, repeats=5):
    """Return the FASTEST of several timings (robust against OS hiccups)."""
    best = float("inf")
    for _ in range(repeats):
        start = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - start)
    return best


data = np.arange(1_000_000, dtype=float)      # a million doubles

t_loop = best_of(python_sum, data)            # interpreted loop
t_builtin = best_of(sum, data)                # Python's built-in sum (still a loop!)
t_numpy = best_of(np.sum, data)               # single compiled call

print(f"python loop : {t_loop * 1000:9.2f} ms")
print(f"builtin sum : {t_builtin * 1000:9.2f} ms")
print(f"np.sum      : {t_numpy * 1000:9.3f} ms")
print(f"speed-up    : ~{t_loop / t_numpy:,.0f}x")

# In Jupyter, `%timeit np.sum(data)` gives you these statistics automatically.

python loop :    120.83 ms
builtin sum :     76.43 ms
np.sum      :     0.435 ms
speed-up    : ~278x


### Antipattern: growing arrays inside loops

Arrays live in fixed buffers, so `np.append` must allocate a FULL new array and copy everything
on EVERY call → quadratic total work. Preallocate, or build a Python list and convert once.

In [21]:
def grow_with_append(n):
    """ANTIPATTERN: each append copies the entire array built so far."""
    out = np.empty(0)
    for i in range(n):
        out = np.append(out, i)           # O(size-so-far) copy EVERY iteration
    return out


def preallocate(n):
    """Right way: one allocation, fill slots by index."""
    out = np.empty(n)
    for i in range(n):
        out[i] = i
    return out


n_growth = 20_000
t_bad = best_of(grow_with_append, n_growth, repeats=3)   # fewer repeats: it's slow on purpose
t_good = best_of(preallocate, n_growth)
print(f"growing appends : {t_bad * 1000:9.1f} ms")
print(f"preallocated    : {t_good * 1000:9.3f} ms  (~{t_bad / max(t_good, 1e-9):,.0f}x faster)")

growing appends :     267.2 ms
preallocated    :     4.545 ms  (~59x faster)


> 💡 **Pro tip:** the performance checklist, in priority order: (1) replace loops with ufunc
expressions, (2) preallocate instead of appending, (3) aggregate once instead of joining
repeatedly, (4) only THEN consider numba/cython. Measure after each step - intuition lies.

## 13. Saving & loading arrays

Binary formats preserve dtype and shape exactly: `.npy` for one array, `.npz` for a labeled
bundle of several. Text formats trade fidelity for human-readability and Excel-friendliness.

In [22]:
scratch = tempfile.gettempdir()                       # OS temp folder - tidy by default

# --- Single array: .npy round trip ---------------------------------------------------
arr_path = os.path.join(scratch, "mlcourse_matrix.npy")
matrix = np.arange(12).reshape(3, 4)                  # int array with known content
np.save(arr_path, matrix)                             # writes exact binary image incl. dtype
loaded = np.load(arr_path)
print("npy identical?", np.array_equal(matrix, loaded), loaded.dtype)

# --- Multiple arrays: .npz archive ----------------------------------------------------
npz_path = os.path.join(scratch, "mlcourse_bundle.npz")
np.savez(npz_path, matrix=matrix, vector=np.linspace(0, 1, 5), label="demo")  # keyword = key
bundle = np.load(npz_path)                            # lazy dict-like container
print("npz keys:", bundle.files)
print("vector from npz:", bundle["vector"])
bundle.close()

# --- Human-readable CSV text round trip ------------------------------------------------
csv_path = os.path.join(scratch, "mlcourse_table.csv")
np.savetxt(csv_path, matrix, delimiter=",", fmt="%d")         # integers as plain text
from_text = np.loadtxt(csv_path, delimiter=",")               # comes back as float64!
print("txt identical after astype?", np.array_equal(matrix, from_text.astype(int)))

for path in (arr_path, npz_path, csv_path):           # leave no litter behind
    os.remove(path)

npy identical? True int64
npz keys: ['matrix', 'vector', 'label']
vector from npz: [0.   0.25 0.5  0.75 1.  ]
txt identical after astype? True


> ⚠️ **Common pitfall:** `loadtxt`/`savetxt` have NO memory of dtypes - integer tables come back
as float64. For anything structural (dtypes, shapes, column names), use `.npy`/`.npz` (or
pandas); reserve CSV for interoperability with the outside world.

> 💡 **Pro tip:** `np.load` on an `.npz` opens LAZILY - members are read from disk only when
indexed, so big bundles stay cheap to inspect. Close the handle (or use a context manager).

## Summary & key takeaways

- Fancy indexing selects by ARBITRARY integer arrays (copies, duplicates OK); use `np.add.at`
  when duplicate indices must accumulate.
- The boolean toolkit: `where` (branch/locate), `clip` (clamp), `count_nonzero` (count),
  `nonzero` (coordinates), `any`/`all` (collapse to one verdict).
- Sort values with `sort`, rank with `argsort`, grab top-k cheaply with `argsort[-k:][::-1]`
  or O(n) `partition`.
- `unique(return_counts=True)` is your frequency table; `intersect1d`/`setdiff1d`/`isin` handle
  membership questions.
- Join with `concatenate`/`vstack`/`hstack`; build feature matrices with `column_stack`;
  `tile` repeats patterns, `repeat` echoes elements; `array_split` tolerates uneven chunks.
- Any formula composed of ufuncs vectorizes for free - distance matrices fall out of the
  `x[:, None] - x[None, :]` broadcasting trick; `np.vectorize` is sugar, not speed.
- Prefer `@` for matrix products and `np.linalg.solve(A, b)` over `inv(A) @ b`.
- Use `np.random.default_rng(seed)` Generators - seeded once, passed around; `np.random.*` is
  the legacy global-state world.
- Slices/reshapes/transposes are views; fancy and boolean selections are copies - prove it to
  yourself with `np.shares_memory` before trusting critical pipelines.
- Vectorize, preallocate, join once, measure with `%timeit`; never grow arrays with `append`.
- `.npy`/`.npz` preserve structure perfectly; text IO is lossy about dtypes.